# Telegram Channel History Backfill (Google Colab)

Beginner-friendly notebook for scanning Telegram channels, parsing media captions, and storing results in MongoDB Atlas with robust handling for large histories.


In [ ]:
# 1) Mount Google Drive for session + output persistence
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/telegram_backfill'
SESSION_DIR = f'{BASE_DIR}/session'
OUTPUT_DIR = f'{BASE_DIR}/output'

import os
os.makedirs(SESSION_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Session dir:', SESSION_DIR)
print('Output dir:', OUTPUT_DIR)


In [ ]:
# 2) Install dependencies
!pip -q install pyrogram tgcrypto pymongo python-dotenv tqdm


In [ ]:
# 3) Imports and helper functions
import json
import os
import re
import time
from datetime import datetime, timezone
from getpass import getpass

from pyrogram import Client
from pyrogram.enums import MessageMediaType
from pyrogram.errors import FloodWait
from pymongo import ASCENDING, MongoClient
from tqdm.auto import tqdm

def get_required_env(name: str, allow_empty: bool = False) -> str:
    value = os.getenv(name)
    if value:
        return value
    value = getpass(f'Enter {name}: ').strip()
    if not value and not allow_empty:
        raise ValueError(f'Missing required environment variable: {name}')
    os.environ[name] = value
    return value

def parse_caption(caption: str | None) -> dict:
    caption = caption or ''

    # Keep original parsing behavior
    kv_pairs = dict(re.findall(r'(?mi)^([a-z0-9_\- ]{2,50}):\s*(.+)$', caption))
    tags = re.findall(r'(?<!\w)#([\w_]+)', caption)

    # New robust extractions
    years = sorted(set(re.findall(r'(?<!\d)((?:19|20)\d{2})(?!\d)', caption)))
    quality_matches = sorted(set(q.lower() for q in re.findall(r'(?i)\b(480p|720p|1080p|2160p)\b', caption)))
    season_episode_matches = sorted(set(s.upper() for s in re.findall(r'(?i)\bS\d{2}E\d{2}\b', caption)))

    return {
        'key_values': {k.strip().lower().replace(' ', '_'): v.strip() for k, v in kv_pairs.items()},
        'tags': sorted(set(t.lower() for t in tags)),
        'years': years,
        'qualities': quality_matches,
        'season_episodes': season_episode_matches,
    }

print('Helpers loaded.')


In [ ]:
# 4) Configure secure environment variables
# Required: TG_API_ID, TG_API_HASH
# MONGODB_URI required only when SAFE_MODE=False

TG_API_ID = int(get_required_env('TG_API_ID'))
TG_API_HASH = get_required_env('TG_API_HASH')

MONGODB_DB = os.getenv('MONGODB_DB', 'telegram_backfill')
MONGODB_COLLECTION = os.getenv('MONGODB_COLLECTION', 'messages')
TG_SESSION_NAME = os.getenv('TG_SESSION_NAME', 'telegram_user_session')

print('Environment configured (secrets hidden).')


In [ ]:
# 5) Backfill configuration
# Examples: '@public_channel', 'https://t.me/channel_name', or numeric chat id
CHANNELS = [
    # '@example_channel',
]

SAFE_MODE = False  # True = dry-run, no MongoDB inserts/updates
START_FROM_MESSAGE_ID = None  # Example: 45000 to skip older messages
PROGRESS_EVERY = 1000
THROTTLE_DELAY = 0.05  # Gentle passive throttling to reduce API pressure

if not CHANNELS:
    print('⚠️ Add at least one channel to CHANNELS before running backfill.')
else:
    print('Channels configured:', CHANNELS)

print('SAFE_MODE:', SAFE_MODE)
print('START_FROM_MESSAGE_ID:', START_FROM_MESSAGE_ID)
print('PROGRESS_EVERY:', PROGRESS_EVERY)
print('THROTTLE_DELAY:', THROTTLE_DELAY)


In [ ]:
# 6) Initialize MongoDB (skipped in SAFE_MODE)
collection = None
if SAFE_MODE:
    print('SAFE_MODE is enabled: MongoDB writes are disabled (dry-run).')
else:
    MONGODB_URI = get_required_env('MONGODB_URI')
    mongo_client = MongoClient(MONGODB_URI)
    collection = mongo_client[MONGODB_DB][MONGODB_COLLECTION]
    collection.create_index(
        [('chat_id', ASCENDING), ('message_id', ASCENDING)],
        unique=True,
        name='uniq_chat_message_id'
    )
    print('MongoDB connected and index ensured.')


In [ ]:
# 7) Run backfill with FloodWait handling + JSONL streaming output
SESSION_PATH = os.path.join(SESSION_DIR, TG_SESSION_NAME)
JSONL_FILE = os.path.join(
    OUTPUT_DIR,
    f"backfill_export_{datetime.now().strftime('%Y%m%d_%H%M%S')}.jsonl"
)

media_types = {
    MessageMediaType.PHOTO: 'photo',
    MessageMediaType.VIDEO: 'video',
    MessageMediaType.DOCUMENT: 'document',
    MessageMediaType.AUDIO: 'audio',
    MessageMediaType.VOICE: 'voice',
    MessageMediaType.ANIMATION: 'animation',
    MessageMediaType.VIDEO_NOTE: 'video_note',
    MessageMediaType.STICKER: 'sticker',
}

with Client(
    name=SESSION_PATH,
    api_id=TG_API_ID,
    api_hash=TG_API_HASH,
    workdir=SESSION_DIR,
    in_memory=False,
) as app, open(JSONL_FILE, 'a', encoding='utf-8') as out_file:
    print('Pyrogram client started.')
    print('JSONL output:', JSONL_FILE)

    global_totals = {
        'processed': 0,
        'media_found': 0,
        'inserted': 0,
        'updated': 0,
        'dry_run': 0,
        'skipped_non_media': 0,
        'skipped_resume': 0,
        'floodwaits': 0,
    }

    for channel in CHANNELS:
        print(f'\n--- Scanning: {channel} ---')
        channel_totals = {
            'processed': 0,
            'media_found': 0,
            'inserted': 0,
            'updated': 0,
            'dry_run': 0,
            'skipped_non_media': 0,
            'skipped_resume': 0,
            'floodwaits': 0,
        }

        offset_id = 0
        stop_channel = False

        while not stop_channel:
            try:
                for msg in app.get_chat_history(channel, offset_id=offset_id):
                    offset_id = msg.id
                    channel_totals['processed'] += 1
                    global_totals['processed'] += 1

                    if START_FROM_MESSAGE_ID is not None and msg.id < int(START_FROM_MESSAGE_ID):
                        channel_totals['skipped_resume'] += 1
                        global_totals['skipped_resume'] += 1
                        print('Resume boundary reached. Stopping channel scan.')
                        stop_channel = True
                        break

                    if not msg.media or msg.media not in media_types:
                        channel_totals['skipped_non_media'] += 1
                        global_totals['skipped_non_media'] += 1
                    else:
                        channel_totals['media_found'] += 1
                        global_totals['media_found'] += 1

                        media_type = media_types[msg.media]
                        media_obj = getattr(msg, media_type, None)
                        chat_id = msg.chat.id if msg.chat else None
                        channel_username = msg.chat.username if msg.chat else None

                        doc = {
                            'chat_id': chat_id,
                            'channel_username': channel_username,
                            'input_channel': str(channel),
                            'message_id': msg.id,
                            'date': msg.date.astimezone(timezone.utc).isoformat() if msg.date else None,
                            'media_type': media_type,
                            'file_id': getattr(media_obj, 'file_id', None),
                            'file_unique_id': getattr(media_obj, 'file_unique_id', None),
                            'caption_raw': msg.caption or '',
                            'caption_meta': parse_caption(msg.caption),
                            'views': getattr(msg, 'views', None),
                            'forwards': getattr(msg, 'forwards', None),
                            'collected_at': datetime.now(timezone.utc).isoformat(),
                        }

                        # Incremental JSON Lines write (memory-safe)
                        out_file.write(json.dumps(doc, ensure_ascii=False) + '\n')

                        if SAFE_MODE:
                            channel_totals['dry_run'] += 1
                            global_totals['dry_run'] += 1
                        else:
                            result = collection.update_one(
                                {'chat_id': doc['chat_id'], 'message_id': doc['message_id']},
                                {'$set': doc},
                                upsert=True,
                            )
                            if result.upserted_id is not None:
                                channel_totals['inserted'] += 1
                                global_totals['inserted'] += 1
                            elif result.modified_count > 0:
                                channel_totals['updated'] += 1
                                global_totals['updated'] += 1

                    if channel_totals['processed'] % PROGRESS_EVERY == 0:
                        print(
                            f"[{channel}] processed={channel_totals['processed']} media={channel_totals['media_found']} "
                            f"inserted={channel_totals['inserted']} updated={channel_totals['updated']} "
                            f"dry_run={channel_totals['dry_run']} skipped_resume={channel_totals['skipped_resume']} "
                            f"floodwaits={channel_totals['floodwaits']}"
                        )

                    # Passive throttling
                    time.sleep(THROTTLE_DELAY)

                # If we finished the for-loop naturally, channel history is exhausted.
                break

            except FloodWait as fw:
                wait_seconds = max(int(getattr(fw, 'value', 1) or 1), 1)
                channel_totals['floodwaits'] += 1
                global_totals['floodwaits'] += 1
                print(f'[FloodWait] {channel}: sleeping {wait_seconds}s ...')
                time.sleep(wait_seconds)
                continue

        print(f"Channel done: {channel}")
        print(channel_totals)

print('\nBackfill complete! Global totals:')
print(global_totals)
print('JSONL saved to:', JSONL_FILE)


## Notes
- `SAFE_MODE=True` lets you test parsing and export without touching MongoDB.
- `START_FROM_MESSAGE_ID` can help resume large scans.
- Output file is JSONL for better memory usage on long runs.
- `THROTTLE_DELAY` adds gentle passive throttling for stability on large channels.
